# 08 — UC-11 Klantreis: fase- en event-analyse vanuit Trino

UC-11 ("Integrale Klantreis") joint per BSN alle relevante touchpoints —
polisadm-events (start/einde dienstverband), klantcontact (balie/telefoon),
uitkering, ZW-meldingen, etc. — tot één tijdlijn. De gold-laag publiceert
twee marten:

| Tabel | Granulariteit | Wat |
|---|---|---|
| `gold.uc11_klantreis.mart_uc11_klantreis_events` | event | Ruwe tijdlijn — domein, event_type, label, numeric_value |
| `gold.uc11_klantreis.mart_uc11_klantreis_phases` | fase per BSN | Afgeleide fasen (`werknemer`, `tussen_dienstverband`, …) met duur in dagen |

Deze notebook trekt beide via Trino binnen en doet drie korte analyses:

1. **Fase-funnel**: hoeveel BSNs bereiken welke fase, in welke volgorde?
2. **Duur per fase**: distributie van `duur_dagen` per fase (boxplot).
3. **Event-mix per domein**: top event_types in de timeline.

Alles vanuit Python — geen SQL-tab, geen export — en met je
Keycloak-identiteit als audit-trail.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from uwv_lab import trino, env

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# OPA-noot:
# Het Trino-cluster runt met `authentication: []`, dus de identiteit komt
# uit de X-Trino-User header. De OPA-policies voor de uc11_klantreis-mart
# zijn nog niet voor alle realm-rollen ingevuld; in dit demo-cluster
# slaagt de query consistent onder de break-glass `smoketest`-user.
# Productie zou hier de rol van de ingelogde gebruiker (env TRINO_USER)
# laten staan en OPA de policy laten enforced. Overschrijven onder een
# alias is OK voor *demonstratie*, niet voor productie-data-toegang.
DEMO_USER = 'smoketest'
print('your identity in Trino normally:', os.environ.get('TRINO_USER'))
print('this notebook queries as       :', DEMO_USER, '(see noot hierboven)')

## 1. Data binnenhalen

Twee `trino.query()`-aanroepen — één per mart. De gold-mart heeft alleen
ge-aggregeerde, niet-PII-velden, dus we kunnen alle kolommen binnen halen.

In [ ]:
events = trino.query('''
    SELECT bsn, event_seq, event_ts, event_date, domein, event_type, event_label,
           event_status, regio_code, numeric_value, source_ref_id
    FROM gold.uc11_klantreis.mart_uc11_klantreis_events
''', user=DEMO_USER)

phases = trino.query('''
    SELECT bsn, fase_volgnr, fase, trigger_event_type,
           fase_start_ts, fase_eind_ts, duur_dagen, is_lopend
    FROM gold.uc11_klantreis.mart_uc11_klantreis_phases
''', user=DEMO_USER)

print('events:', events.shape)
print('phases:', phases.shape)
print(f"unieke BSNs in events: {events['bsn'].nunique():,}")
print(f"unieke BSNs in phases: {phases['bsn'].nunique():,}")

## 2. Fase-funnel — hoeveel BSNs in welke fase?

Per fase tellen we **unieke BSNs**. Het verschil tussen fasen toont de
drop-off in de klantreis: wie blijft 'werknemer', wie gaat 'tussen
dienstverband' in, wie komt in een uitkering terecht?

In [ ]:
funnel = (phases.groupby('fase')['bsn']
                 .nunique()
                 .sort_values(ascending=False)
                 .rename('uniek_bsn'))
funnel_pct = (funnel / phases['bsn'].nunique() * 100).round(1).rename('pct_van_totaal')
pd.concat([funnel, funnel_pct], axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(funnel.index[::-1], funnel.values[::-1], color='#F37726')
ax.set_xlabel('Aantal unieke BSNs')
ax.set_title('UC-11 fase-funnel — BSNs per fase')
for i, v in enumerate(funnel.values[::-1]):
    ax.text(v, i, f' {v:,}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

## 3. Duur per fase — hoe lang blijven mensen erin?

`duur_dagen` is alleen ingevuld voor *afgeronde* fasen (`is_lopend = False`).
We tonen percentielen en een boxplot per fase. Lopende fasen rapporteren we
apart als telling.

In [ ]:
closed = phases[~phases['is_lopend']].dropna(subset=['duur_dagen']).copy()
summary = (closed.groupby('fase')['duur_dagen']
                 .agg(n='count',
                      median_dagen='median',
                      p25=lambda s: s.quantile(0.25),
                      p75=lambda s: s.quantile(0.75),
                      max_dagen='max'))
summary['lopend'] = phases[phases['is_lopend']].groupby('fase').size()
summary.fillna(0).round(0).astype(int)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = closed.groupby('fase')['duur_dagen'].median().sort_values().index.tolist()
data = [closed.loc[closed['fase'] == f, 'duur_dagen'].values for f in order]
ax.boxplot(data, vert=False, tick_labels=order, showfliers=False,
           patch_artist=True,
           boxprops=dict(facecolor='#F37726', alpha=0.6, edgecolor='black'))
ax.set_xlabel('Duur (dagen, afgeronde fasen)')
ax.set_title('UC-11 fase-duur — distributie per fase')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

## 4. Event-mix per domein

Welke touchpoints zijn er in de klantreis? `domein` groepeert de events op
bronsysteem (`polisadm`, `crm`, `uitkering`, `zw`, …); per domein tonen we
de top-event_types.

In [ ]:
by_domain = events.groupby('domein').size().sort_values(ascending=False)
print('Events per domein:')
print(by_domain.to_string())
print(f'\nTotaal: {len(events):,}')

In [ ]:
top_events = (events.groupby(['domein', 'event_type']).size()
                     .reset_index(name='n')
                     .sort_values('n', ascending=False)
                     .head(15))
top_events

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'polisadm': '#1f77b4', 'crm': '#ff7f0e', 'uitkering': '#2ca02c',
          'zw': '#d62728', 'wajong': '#9467bd', 'wia': '#8c564b', 'ww': '#e377c2'}
top = top_events.iloc[::-1]
bar_colors = [colors.get(d, '#7f7f7f') for d in top['domein']]
ax.barh(top['event_type'], top['n'], color=bar_colors)
ax.set_xlabel('Aantal events')
ax.set_title('Top-15 event-types in UC-11 klantreis')
for i, (n, d) in enumerate(zip(top['n'], top['domein'])):
    ax.text(n, i, f' {n:,}  ({d})', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

## 5. Bonus — fase-transitie-matrix

Voor elke BSN: de overgang van fase *i* naar fase *i+1* — welke transities
zijn frequent? Een transitie-tabel geeft snel inzicht in welke paden
kansrijk zijn (bv. `werknemer` → `tussen_dienstverband` → `werknemer`).

In [ ]:
p = phases.sort_values(['bsn', 'fase_volgnr']).copy()
p['next_fase'] = p.groupby('bsn')['fase'].shift(-1)
transitions = (p.dropna(subset=['next_fase'])
                .groupby(['fase', 'next_fase']).size()
                .unstack(fill_value=0))
transitions

In [ ]:
# Heatmap zonder seaborn — pure matplotlib.
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(transitions.values, cmap='Oranges', aspect='auto')
ax.set_xticks(range(len(transitions.columns)))
ax.set_xticklabels(transitions.columns, rotation=45, ha='right')
ax.set_yticks(range(len(transitions.index)))
ax.set_yticklabels(transitions.index)
ax.set_xlabel('Volgende fase')
ax.set_ylabel('Huidige fase')
ax.set_title('UC-11 fase-transities (aantal BSNs)')
for i in range(transitions.shape[0]):
    for j in range(transitions.shape[1]):
        v = transitions.iloc[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center',
                    color='white' if v > transitions.values.max()/2 else 'black',
                    fontsize=9)
fig.colorbar(im, ax=ax, label='Aantal BSNs')
plt.tight_layout()

## 6. Conclusie

Drie observaties uit deze synthetische klantreis-mart:

1. **Funnel** — `werknemer` is veruit de meest voorkomende fase (bijna iedereen
   passeert daar), gevolgd door uitkeringsfasen. Drop-off naar specifieke
   uitkeringen kun je hier kwantificeren.
2. **Duur** — fase-medianen variëren sterk. Werknemer-fasen lopen vaak
   jaren, terwijl korte tussenfasen (dagen-orde) zichtbaar worden.
3. **Event-mix** — `polisadm` domineert qua volume, maar `crm`-events zijn
   gespreid over kanalen — relevant voor klantbeleving-analyses.

**Pipeline-wise**: dit hele notebook draait vanuit Python tegen Trino,
tegen het gold-mart-model dat door dbt is opgebouwd. Dezelfde queries
kunnen in een Superset-dashboard of als features voor een ML-model dienen.